# Coordinator and Subagents (Managed Agents API)

> **CCA-F Domain 1 - Agentic (27%).** The exam tests **isolated-context orchestration**: a **coordinator** spawns each specialist in its **own session thread** with its **own fresh context**, and only the specialist's result flows back. The trap is the **daisy-chain anti-pattern** - piping every raw specialist transcript through one ever-growing coordinator context until it blows the window and the reasoning rots.

**The mental model: a newsroom.** Think of an **editor** who is handed a story to write. The editor does not do the legwork. The editor **assigns** it: one **reporter** digs up the facts, another **reporter** gathers the quotes. Each reporter works alone, does not see the other's notebook, and hands back only a finished summary. The editor then writes the final piece from those two summaries.

That org chart *is* the coordinator/subagent pattern:

```
        Editor  (coordinator)  <- you talk to this one
        /      \
  Facts        Quotes          (subagents / specialists)
 reporter      reporter        each in its own isolated thread
```

This notebook builds exactly that: a coordinator plus a **facts reporter** and a **quotes reporter**. We run **one delegated turn** and print the stream as an **indented tree** so the parent/child relationship is visible on screen - the editor's lines flush-left, each reporter's lines indented under its name.

> **These calls are live, billable, and beta-gated.** Run them only with Managed Agents beta access.

### 1. Install dependencies

The `anthropic` SDK carries the Managed Agents surface under `client.beta.agents`, `.environments`, and `.sessions`. `python-dotenv` reads the API key from `.env`.

In [1]:
# uv-managed venvs ship WITHOUT pip, so `%pip install` fails here.
# Use uv instead, pointed at THIS kernel's interpreter so packages land
# where the kernel actually looks. Idempotent: uv no-ops if already satisfied.
import subprocess, sys

subprocess.run(
    ["uv", "pip", "install", "--python", sys.executable, "anthropic", "python-dotenv"],
    check=True,
)
print("Dependencies ready.")

Dependencies ready.


### 2. Load environment variables and create the client

`load_dotenv()` reads the API key from `.env`. The Managed Agents surface is **beta-gated**, so we set `BETA` once and pass `betas=[BETA]` on every call. `MODEL` stays on **Haiku 4.5** (the repo default); a coordinator plus two specialists at production quality does not need Sonnet.

In [2]:
import os
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
MODEL = "claude-haiku-4-5"            # unversioned alias -> current Haiku 4.5 snapshot
BETA = "managed-agents-2026-04-01"    # Managed Agents beta header, passed on every call

# Teardown guard. Default OFF so "Run All" in class leaves the editor, both
# reporters, and the session live for Console inspection. Headless smoke tests
# set ARCHIVE_ON_RUN=1 to force cleanup so nothing dangles in CI. See final cell.
RUN_TEARDOWN = os.environ.get("ARCHIVE_ON_RUN", "0") == "1"

### 3. Build the reporter roster (the subagents)

Each reporter is a **plain agent** - no `multiagent` config (the depth limit is 1, so a subagent cannot itself be a coordinator). Give each a **narrow `system` prompt** so its context stays focused on its one job. When the editor spawns one, that reporter runs in its **own session thread with fresh context**; the facts reporter never sees the quotes reporter's notes, and vice versa. That isolation is the whole point of the pattern - and the reason the editor's context does not balloon.

In [3]:
# Two narrow reporters. Each is a normal agent (no multiagent config).
facts_reporter = client.beta.agents.create(
    model={"id": MODEL},
    name="facts-reporter",
    system=(
        "You are a facts reporter. Given a story topic, return 3 to 4 concrete, "
        "verifiable facts as a short bullet list. Facts only - no quotes, no opinion."
    ),
    betas=[BETA],
)

quotes_reporter = client.beta.agents.create(
    model={"id": MODEL},
    name="quotes-reporter",
    system=(
        "You are a quotes reporter. Given a story topic, return 2 short illustrative "
        "quotes with a plausible speaker and role for each. Quotes only - no fact list."
    ),
    betas=[BETA],
)

print("facts reporter: ", facts_reporter.id)
print("quotes reporter:", quotes_reporter.id)

facts reporter:  agent_0161FDxRvdgboN5zXVgHyS6L
quotes reporter: agent_01RYNej3YVa8AoUaEYT5CJnd


### 4. Wire the editor with the `multiagent` config (the coordinator)

The `multiagent` param turns an agent into a **coordinator** - our **editor**. Its shape is verified against the SDK (`BetaManagedAgentsMultiagentParams`): two required keys.

| Key | Value | Notes |
|---|---|---|
| `type` | `"coordinator"` | The only allowed value. |
| `agents` | roster list, 1-20 entries | Each entry is an **agent-ID string** (simplest), a versioned `{"type":"agent","id":...,"version":...}` ref, or `{"type":"self"}` for recursion. Referenced agents must exist, must not be archived, and must not themselves set `multiagent` (**depth limit 1**). |

We use the **string form** - the simplest roster entry - and stay minimal: `type` + `agents` only. The editor's `system` prompt is the assignment brief: it splits the story into a facts question and a quotes question, hands each to the matching reporter in an isolated thread, then writes the final piece from what comes back.

In [4]:
# The multiagent config is the coordinator shape. Roster entries are the
# reporter agent IDs (string form). Depth limit is 1, so neither reporter
# may itself carry a multiagent config.
editor = client.beta.agents.create(
    model={"id": MODEL},
    name="newsroom-editor",
    system=(
        "You are a newsroom editor. Break the assignment into a facts question and a "
        "quotes question, delegate the facts question to the facts reporter and the "
        "quotes question to the quotes reporter, then write a short 2-paragraph story "
        "that weaves their material together. Delegate the legwork; do not gather the "
        "facts or invent the quotes yourself."
    ),
    multiagent={
        "type": "coordinator",
        "agents": [facts_reporter.id, quotes_reporter.id],
    },
    betas=[BETA],
)

print("editor (coordinator):", editor.id)

editor (coordinator): agent_019ZsePhAQCxHBRpCi5nH4uX


### 5. Run one delegated turn - and watch the parent/child dialogue

Same spine as any Managed Agents session: **create an environment**, **create a session** bound to the editor, **send** the user turn, then **stream** events until the session goes idle.

The key to *seeing* the hierarchy is knowing which events reach the primary stream. A subagent's own `agent.message` events stay on **its** thread; they do not cross-post. What crosses to the primary stream is the **agent-to-agent messaging pair** - the actual editor/reporter dialogue:

| Event | Meaning | Useful field |
|---|---|---|
| `session.thread_created` | editor spawned a reporter | `agent_name`, `session_thread_id` |
| `agent.thread_message_sent` | **editor -> reporter**: the assignment | `to_session_thread_id`, `content` |
| `agent.thread_message_received` | **reporter -> editor**: the finished material | `from_session_thread_id`, `content` |
| `agent.message` (empty thread id) | the **editor** speaking on the primary thread | `content` |
| `session.status_idle` | turn over | `stop_reason` |

We keep a small map of `thread_id -> reporter name` from the `thread_created` events, then render an indented tree: editor lines flush-left, and each reporter's inbound assignment and outbound material indented under that reporter's name. That is the org chart, live. As always, **stop on `session.status_idle`** - it carries `stop_reason`, not the prose. Stopping on vibes is the exam trap.

In [5]:
# Environment + session bound to the editor (coordinator).
env = client.beta.environments.create(name="newsroom-scratch", betas=[BETA])
session = client.beta.sessions.create(
    agent=editor.id,              # agent accepts the agent ID string
    environment_id=env.id,
    betas=[BETA],
)

# Send one assignment.
client.beta.sessions.events.send(
    session.id,
    events=[{
        "type": "user.message",
        "content": [{
            "type": "text",
            "text": "Write a short story about a city opening its first public rooftop park.",
        }],
    }],
    betas=[BETA],
)


def text_of(content) -> str:
    """Flatten an event's content blocks into plain text."""
    return "".join(b.text for b in content if getattr(b, "type", None) == "text").strip()


def emit(label: str, text: str, *, child: bool) -> None:
    """Editor lines flush-left; reporter dialogue indented so the hierarchy is visible."""
    if not text:
        return
    head = "    +-- " if child else ""
    indent = "       " if child else ""
    body = "\n".join(indent + line for line in text.splitlines())
    print(f"{head}[{label}]\n{body}\n")


# thread_id -> reporter name, filled in as the editor spawns each child thread.
thread_names: dict[str, str] = {}

for event in client.beta.sessions.events.stream(session.id, betas=[BETA]):
    etype = event.type

    if etype == "session.thread_created":
        # The editor just handed the story to a reporter. Record who, and announce it.
        thread_names[event.session_thread_id] = event.agent_name
        print(f"    >>> EDITOR spawns child thread: {event.agent_name}\n")

    elif etype == "agent.thread_message_sent":
        # Editor -> reporter: the assignment the child was handed.
        who = thread_names.get(event.to_session_thread_id, "reporter")
        emit(f"EDITOR asks -> {who}", text_of(event.content), child=True)

    elif etype == "agent.thread_message_received":
        # Reporter -> editor: the finished material coming back up.
        who = thread_names.get(event.from_session_thread_id, "reporter")
        emit(f"{who} answers", text_of(event.content), child=True)

    elif etype == "agent.message":
        # Empty thread id => the editor speaking on the primary thread.
        if not (getattr(event, "session_thread_id", "") or ""):
            emit("EDITOR (coordinator)", text_of(event.content), child=False)

    elif etype == "session.status_idle":
        # The ONLY correct stop. stop_reason lives here, not on the prose.
        print("idle - stop_reason:", getattr(event, "stop_reason", None))
        break

    >>> EDITOR spawns child thread: facts-reporter

    +-- [EDITOR asks -> facts-reporter]
       Research and compile the key facts about a city's first public rooftop park opening. Include details such as: the city and park name, location/address, size of the park, what amenities and features it includes (gardens, seating areas, walking paths, views, etc.), cost of the project, how long it took to build, the building(s) it's on top of, any environmental or sustainability features, expected visitor capacity, and when it opened. Create realistic, plausible details suitable for a news story about this type of urban development project.



    >>> EDITOR spawns child thread: quotes-reporter

    +-- [EDITOR asks -> quotes-reporter]
       Gather quotes from people involved in or affected by a city's first public rooftop park opening. Obtain 3-4 substantive quotes from different perspectives, such as: a city official or mayor, the project architect or designer, a local resident or community member, and/or a parks department director. Quotes should reflect excitement about the project, specific details about its impact, and diverse viewpoints on what it means for the city. Create realistic, plausible quotes suitable for a news story.



[EDITOR (coordinator)]
I've sent both reporters to work on this story. The facts reporter is gathering details about the rooftop park project, and the quotes reporter is collecting perspectives from key people involved. Once they report back, I'll weave their material into a compelling 2-paragraph story. Check back shortly!



    +-- [facts-reporter answers]
       I need clarification on the scope of this task. My role is to report concrete, verifiable facts. However, the request asks me to "create realistic, plausible details" for a fictional rooftop park story.
       
       Are you asking me to:
       1. Research and report facts about an actual, real rooftop park that opened (e.g., a specific city's first public rooftop park)?
       2. Generate fictional but plausible details for a hypothetical rooftop park as if I were writing fiction/a made-up news story?
       
       These are different tasks. Please clarify which you need so I can proceed appropriately.



    +-- [EDITOR asks -> facts-reporter]
       Good question. Go with option 2: generate fictional but plausible details for a hypothetical rooftop park as if you were writing a news story. Develop specific, credible facts that would work in a news narrative—city name, park name, location, size, amenities, cost, timeline, building details, environmental features, etc. Make it all realistic and internally consistent, the way a reporter would if covering a real urban development project. Your job is to create the factual framework that will make this story feel authentic and grounded.

    +-- [quotes-reporter answers]
       I'm ready to create quotes for a rooftop park opening story. Before I proceed, I need clarification: my system role specifies 2 short illustrative quotes, but your task asks for 3-4 substantive quotes from different perspectives. Should I deliver 2 or 3-4 quotes? And should they be short and illustrative, or longer and substantive? Please confirm the exact scope so 

    +-- [EDITOR asks -> quotes-reporter]
       Go with the task scope I gave you: 3-4 substantive quotes from different perspectives. These should be detailed and meaningful—not one-liners, but full sentences or short paragraphs that capture real perspective and insight. Think of them as the kind of quotes that would anchor different angles of a news story (city official, architect/designer, resident/community member, etc.). Substantive means they should add real information and voice to the narrative, not just cheerleading.



    +-- [facts-reporter answers]
       Here are the key facts for a fictional but plausible rooftop park news story:
       
       • **Park name and location**: Riverside Commons Rooftop Park, located at 1450 South Platte River Drive in Denver's Santa Fe Arts District, spans 3.2 acres atop the mixed-use Riverside Commons development (8 stories containing 120,000 sq ft of office, 45,000 sq ft of retail, and 240 residential units)
       
       • **Amenities and features**: Native plant gardens (1.8 acres of wildflower meadows and native shrubs), elevated walking paths with downtown skyline views, 12 seating clusters with shade structures, 0.5-acre dog park with separated small/large dog areas, outdoor fitness equipment, community gathering space with performance lawn, and irrigation-integrated rain gardens
       
       • **Cost and timeline**: $27 million project budget; construction began May 2021 and completed April 2024 (36-month timeline); project managed by landscape architect

[EDITOR (coordinator)]
Excellent facts are in. Now waiting for the quotes reporter to deliver the key voices from this story.



    +-- [quotes-reporter answers]
       Here are 4 substantive quotes from different perspectives on the city's first public rooftop park opening:
       
       **Mayor Sarah Chen, City Administration:**
       "This park represents a fundamental shift in how we think about urban density. We've unlocked 2.3 acres of space in the heart of the city without consuming a single square foot of ground-level land. What excites me most isn't just the green space itself—it's that this project proves that cities can grow vertically and sustainably at the same time. It's a template we can replicate across 15 other rooftops in the downtown corridor."
       
       **David Liu, Lead Architect, Liu + Park Design Studio:**
       "The engineering challenge was significant: waterproofing, drainage, weight distribution on a structure built in 1987. We had to reinforce the underlying concrete while keeping costs reasonable. But the real design innovation was creating microclimates—we installed wind ba

[EDITOR (coordinator)]
# Denver Opens Its First Public Rooftop Park, Reclaiming the Urban Sky

Denver opened Riverside Commons Rooftop Park to the public last month, unveiling a 3.2-acre green space perched 12 stories above the South Platte River corridor—the city's first publicly accessible rooftop park. The $27 million project, which crowns a mixed-use development in the Santa Fe Arts District, transforms what would have been unused concrete into wildflower meadows, walking paths, a dog park, and community gathering spaces. The park's design thoughtfully weaves sustainability into every detail: a green roof system that captures 60% of annual rainfall, a 250-kilowatt solar array powering its lighting and irrigation, and native plantings selected to attract pollinators and withstand drought. "This park represents a fundamental shift in how we think about urban density," Mayor Sarah Chen said. "We've unlocked 2.3 acres of space in the heart of the city without consuming a single square 

### 6. Tear down (guarded)

Always **archive** what you created so no billable session dangles. There is no `agents.delete` - archive is the retirement path for both agents and sessions. Archive the session first, then the editor, then both reporters.

**This cell is guarded by `RUN_TEARDOWN`** (set in the client cell above). It defaults to **OFF** so a "Run All" during class leaves the coordinator, both reporters, and the session **live** - handy for inspecting the delegation threads in the Claude Console after the demo. To archive, flip the switch and re-run this one cell, or set `ARCHIVE_ON_RUN=1` before launching (how the smoke tests force cleanup).

In [ ]:
if RUN_TEARDOWN:
    # Archive the session, then every agent. No agents.delete exists.
    client.beta.sessions.archive(session.id, betas=[BETA])
    client.beta.agents.archive(editor.id, betas=[BETA])
    client.beta.agents.archive(facts_reporter.id, betas=[BETA])
    client.beta.agents.archive(quotes_reporter.id, betas=[BETA])
    print("Archived session + 3 agents. Nothing left billable.")
else:
    print("Teardown SKIPPED - session and all 3 agents are still LIVE.")
    print("Archive manually: flip RUN_TEARDOWN or set ARCHIVE_ON_RUN=1 and re-run.")
    print("  editor:          ", editor.id)
    print("  facts reporter:  ", facts_reporter.id)
    print("  quotes reporter: ", quotes_reporter.id)
    print("  session:         ", session.id)